In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import nltk

This project uses "The Blog Authorship Corpus"https://www.kaggle.com/rtatman/blog-authorship-corpus

The Blog Authorship Corpus consists of the collected posts of 19,320 bloggers gathered from blogger.com in August 2004. 
The corpus incorporates a total of 681,288 posts and over 140 million words - or approximately 35 posts and 
7250 words per person.  

J. Schler, M. Koppel, S. Argamon and J. Pennebaker (2006). Effects of Age and Gender on Blogging in Proceedings of 2006
AAAI Spring Symposium on Computational Approaches for Analyzing Weblogs. 

### 1 LOAD THE DATASET

In [2]:
blog_data = pd.read_csv("blogtext.csv")
blog_data = blog_data.head(10000)

In [3]:
blog_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 7 columns):
id        10000 non-null int64
gender    10000 non-null object
age       10000 non-null int64
topic     10000 non-null object
sign      10000 non-null object
date      10000 non-null object
text      10000 non-null object
dtypes: int64(2), object(5)
memory usage: 547.0+ KB


In [4]:
blog_data.describe()

,id,age
count,1.000000e+04,10000.000000
mean,1.854823e+06,28.019300
std,1.303245e+06,8.123923
min,4.677050e+05,13.000000
25%,6.497900e+05,23.000000
50%,1.103575e+06,27.000000
75%,3.176655e+06,35.000000
max,4.321554e+06,46.000000


In [5]:
blog_data.describe(include='object')

,gender,topic,sign,date,text
count,10000,10000,10000,10000,10000
unique,2,26,12,718,9949
top,male,indUnk,Aries,"05,August,2004",
freq,5916,3287,4198,2329,13


In [6]:
blog_data.shape

(10000, 7)

In [7]:
# Get some idea about text
blog_data['text'].head(50)

0                Info has been found (+/- 100 pages,...
1                These are the team members:   Drewe...
2                In het kader van kernfusie op aarde...
3                      testing!!!  testing!!!          
4                  Thanks to Yahoo!'s Toolbar I can ...
5                  I had an interesting conversation...
6                  Somehow Coca-Cola has a way of su...
7                  If anything, Korea is a country o...
8                  Take a read of this news article ...
9                  I surf the English news sites a l...
10                 Ah, the Korean language...it look...
11                 If you click on my profile you'll...
12                 Last night was pretty fun...mostl...
13                 There is so much that is differen...
14                  urlLink    Here it is, the super...
15                 One thing I love about Seoul (and...
16                  urlLink    Wonderful oh-gyup-sal...
17                 Here is the latest from the K

In [8]:
blog_data['text'].tail(50)

9950                     Oh, man, Amanda, I don't not ...
9951                     I'm so so sorry, Amanda!&nbsp...
9952                     No, no, no, no!&nbsp; I promi...
9953                     As in Sucky McSuckmeister.&nb...
9954                     Wow, other people are funny! ...
9955                     I'm randomly reading other pe...
9956                     this entry is for the sole pu...
9957                     goddamn, Geoff.&nbsp;&nbsp;If...
9958                     this is my commenting entry.&...
9959                     I've been thinking about all ...
9960                     I don't understand you, Geoff...
9961                     This morning it was marachino...
9962                     Ì'm finally hanging out with ...
9963                     Get your mind out of the gutt...
9964                     My big brother, Ucre is givin...
9965                     does anybody know why shoulde...
9966                     I just looked at Devin's prof...
9967          

### 2 Preprocessing Rows Of Text Column

a. Remove unwanted characters
b. Convert text to lowercase
c. Remove unwanted spaces
d. Remove stopwords

In [9]:
import re

# function for text cleaning 
def preprocess_text(text):
    # a
    # remove everything except alphabets 
    text = re.sub("[^a-zA-Z]"," ",text) 
    
    # b convert text to lowercase 
    text = text.lower() 
    
    # c remove whitespaces 
    text = ' '.join(text.split()) 
    
    #d #remove stop words
    from nltk.corpus import stopwords
    stop = set(stopwords.words('english'))
    stop.add("urllink")
                 
    text = ' '.join([word for word in text.split() if word not in (stop)])

    #remove backslash-apostrophe 
    #NOTE: removing after stop word
    text = re.sub("\'", "", text) 
    
    return text

In [10]:
final_blog_data = blog_data.copy()
final_blog_data['text'] = blog_data['text'].apply(lambda x: preprocess_text(x))

In [11]:
#Check preprocessed text
# Get some idea about text
final_blog_data['text'].head(50)

0     info found pages mb pdf files wait untill team...
1     team members drewes van der laag mail ruiyu xi...
2     het kader van kernfusie op aarde maak je eigen...
3                                       testing testing
4     thanks yahoo toolbar capture urls popups means...
5     interesting conversation dad morning talking k...
6     somehow coca cola way summing things well earl...
7     anything korea country extremes everything see...
8     take read news article joongang ilbo north kor...
9     surf english news sites lot looking tidbits ko...
10    ah korean language looks difficult first figur...
11    click profile make startling discovery born ye...
12    last night pretty fun mostly company kept rece...
13    much different anything ever seen well travell...
14    superfantastic phonebox today great day lovely...
15    one thing love seoul mean korea general happen...
16    wonderful oh gyup sal favorite pork restaurant...
17    latest korean rumor mill made way coquitla

In [12]:
final_blog_data['text'].tail(50)

9950    oh man amanda like promise promise promise nbs...
9951    sorry amanda nbsp like whole lot cause amanda ...
9952    nbsp promise like anything amanda nbsp promise...
9953    sucky mcsuckmeister nbsp freaking hate geoff n...
9954    wow people funny among amusing nbsp nbsp nbsp ...
9955    randomly reading people blogs one thirteen yea...
9956    entry sole purpose seeing name recently publis...
9957    goddamn geoff nbsp nbsp wanna prove people mat...
9958    commenting entry nbsp bored simply posting nbs...
9959    thinking passwords online names lately nbsp ho...
9960                           understand geoffrey moddle
9961    morning marachino cherry nbsp sickiningly swee...
9962    finally hanging brianna nbsp whoo hoo feeling ...
9963    get mind gutter nbsp ok randomly spent night g...
9964    big brother ucre giving geoff nbsp saying inac...
9965    anybody know shoulders start hurt using mouse ...
9966    looked devin profile cause told talked bunch f...
9967    event 

### 3. COLUMN MERGING
a. Label columns to merge: “gender”, “age”, “topic”, “sign”
b. After completing the previous step, there should be only two columns in your dataframe i.e. “text” and “labels” as shown in the below image

In [13]:
final_blog_data['labels'] = blog_data[["gender","age","topic","sign"]].astype('str').values.tolist()       
print(final_blog_data['labels'])

0                      [male, 15, Student, Leo]
1                      [male, 15, Student, Leo]
2                      [male, 15, Student, Leo]
3                      [male, 15, Student, Leo]
4       [male, 33, InvestmentBanking, Aquarius]
                         ...                   
9995               [female, 25, indUnk, Pisces]
9996               [female, 25, indUnk, Pisces]
9997               [female, 25, indUnk, Pisces]
9998               [female, 25, indUnk, Pisces]
9999               [female, 25, indUnk, Pisces]
Name: labels, Length: 10000, dtype: object


In [14]:
final_blog_data.head()

,id,gender,age,topic,sign,date,text,labels
0,2059027,male,15,Student,Leo,"14,May,2004",info found pages mb pdf files wait untill team...,"[male, 15, Student, Leo]"
1,2059027,male,15,Student,Leo,"13,May,2004",team members drewes van der laag mail ruiyu xi...,"[male, 15, Student, Leo]"
2,2059027,male,15,Student,Leo,"12,May,2004",het kader van kernfusie op aarde maak je eigen...,"[male, 15, Student, Leo]"
3,2059027,male,15,Student,Leo,"12,May,2004",testing testing,"[male, 15, Student, Leo]"
4,3581210,male,33,InvestmentBanking,Aquarius,"11,June,2004",thanks yahoo toolbar capture urls popups means...,"[male, 33, InvestmentBanking, Aquarius]"


In [15]:
final_blog_data.tail()

,id,gender,age,topic,sign,date,text,labels
9995,1705136,female,25,indUnk,Pisces,"19,May,2004",take home forever may rest sleep arms forgotte...,"[female, 25, indUnk, Pisces]"
9996,1705136,female,25,indUnk,Pisces,"23,June,2004",seductive secretness behind doors warning neve...,"[female, 25, indUnk, Pisces]"
9997,1705136,female,25,indUnk,Pisces,"21,June,2004",kind need holding hand petting hair cry bring ...,"[female, 25, indUnk, Pisces]"
9998,1705136,female,25,indUnk,Pisces,"09,June,2004",blurry outside sounds people mingle pass darkn...,"[female, 25, indUnk, Pisces]"
9999,1705136,female,25,indUnk,Pisces,"07,June,2004",body feels broken mind rejoices thought warmth...,"[female, 25, indUnk, Pisces]"


In [16]:
final_blog_data.describe()

,id,age
count,1.000000e+04,10000.000000
mean,1.854823e+06,28.019300
std,1.303245e+06,8.123923
min,4.677050e+05,13.000000
25%,6.497900e+05,23.000000
50%,1.103575e+06,27.000000
75%,3.176655e+06,35.000000
max,4.321554e+06,46.000000


### 4 Separate features and labels, and split the data into training and testing 

In [17]:
#Separate Features and labels
X = final_blog_data[["text"]]
y = final_blog_data[["labels"]]

# TRain and test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=7 )
print(X_train.shape, y_train.shape)
print(X_test.shape, y_test.shape)
y_train['labels'].head()


(7500, 1) (7500, 1)
(2500, 1) (2500, 1)


4881    [male, 25, BusinessServices, Sagittarius]
89                    [female, 14, indUnk, Aries]
845              [male, 17, Student, Sagittarius]
8338        [female, 25, Technology, Sagittarius]
9947              [female, 16, indUnk, Capricorn]
Name: labels, dtype: object

### 5. Vectorize the features
a. Create a Bag of Words using count vectorizer
i. Use ngram_range=(1, 2)
ii. Vectorize training and testing features
b. Print the term-document matrix

#### Also used min_df, max_df,stemmer and lemmatizer options and chose the best vectorizer. Best vectorizer was with min_df=2, max_df=0.50 and stemming . Comparison was done by checking accuracy of final classification model.

In [18]:
# Before using Countvectorizer, we are going to create a tokenizer using nltk that stems the words for us
from nltk import word_tokenize          

In [19]:
from nltk.stem.porter import PorterStemmer
stemmer = PorterStemmer()
def stem_tokens(tokens, stemmer):
    stemmed = []
    for item in tokens:
        stemmed.append(stemmer.stem(item))
    return stemmed

In [20]:
def tokenize(text):
    tokens = word_tokenize(text)
    stems = stem_tokens(tokens, stemmer)
    return stems

In [21]:
from sklearn.feature_extraction.text import CountVectorizer

In [22]:
# Create base vectorizer
#max_df is used for removing data values that appear too frequently, also known as "corpus-specific stop words".
#max_df = 0.50 means "It ignores terms that appear in more than 50% of the documents".
#min_df is used for removing terms that appear too infrequently(2 means ignore terms that appear in less than 2 documents".)

vectorizer_base = CountVectorizer(ngram_range = (1,2),min_df=2, max_df=0.50)

In [23]:
# Create our vectorizer
vectorizer_stem = CountVectorizer(ngram_range = (1,2), min_df=2, max_df=0.50, tokenizer=tokenize)

In [24]:
from nltk.stem import WordNetLemmatizer 
class LemmaTokenizer:
    def __init__(self):
        self.wnl = WordNetLemmatizer()
    def __call__(self, doc):
         return [self.wnl.lemmatize(t) for t in word_tokenize(doc)]

vectorizer_lemma = CountVectorizer(ngram_range = (1,2), tokenizer=LemmaTokenizer())  

In [25]:
#vectorizer = vectorizer_base
vectorizer = vectorizer_stem
#vectorizer = vectorizer_lemma

In [26]:
vectorizer.fit(X_train.text)

CountVectorizer(analyzer='word', binary=False, decode_error='strict',
                dtype=<class 'numpy.int64'>, encoding='utf-8', input='content',
                lowercase=True, max_df=0.5, max_features=None, min_df=2,
                ngram_range=(1, 2), preprocessor=None, stop_words=None,
                strip_accents=None, token_pattern='(?u)\\b\\w\\w+\\b',
                tokenizer=<function tokenize at 0x000002109773A5E8>,
                vocabulary=None)

In [27]:
# summarize
#print(vectorizer.vocabulary_)

In [28]:
X_train_dtm = vectorizer.transform(X_train.text)
print ('Shape of Sparse Matrix: ', X_train_dtm.shape)
print ('Amount of Non-Zero occurences: ', X_train_dtm.nnz)
print ('sparsity: %.2f%%' % (100.0 * X_train_dtm.nnz /
                             (X_train_dtm.shape[0] * X_train_dtm.shape[1])))

Shape of Sparse Matrix:  (7500, 79915)
Amount of Non-Zero occurences:  685842
sparsity: 0.11%


In [29]:
X_test_dtm = vectorizer.transform(X_test.text)

In [30]:
print(X_train_dtm.shape)
print(X_test_dtm.shape)

(7500, 79915)
(2500, 79915)


In [31]:
# examine the vocabulary and document-term matrix together, printing only 25 rows to avoid memory error
pd.DataFrame(X_train_dtm.toarray()[0:25], columns=(vectorizer.get_feature_names()))

,aa,aaa,aaah,aal,aal elimin,aaron,aaron like,ab,aba,aback,...,zone natur,zoo,zoo anim,zookx,zooland,zoom,zy,zzz,zzzz,zzzzz
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
7,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
9,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [32]:
# last 50 features
#print(vectorizer.get_feature_names()[-100:])

### 6 Create a dictionary 
To get the count of every label i.e. the key will be label name and value will be the total count of the label.

In [33]:
all_labels = sum(final_blog_data['labels'],[])
labels_frequency = nltk.FreqDist(all_labels)
unique_labels = set(all_labels)

final_dict = {}

for key in unique_labels:
    final_dict[key] = labels_frequency[key]
    
# Printing dictionary 
print ("Label dictionary is : " + str(final_dict)) 

Label dictionary is : {'Fashion': 1622, 'Law': 11, '33': 136, '17': 1185, 'Capricorn': 215, 'Education': 270, 'Banking': 16, 'Internet': 118, 'Museums-Libraries': 17, '13': 42, 'Leo': 301, '38': 46, 'LawEnforcement-Security': 10, 'Arts': 45, 'Scorpio': 971, '37': 33, 'Telecommunications': 2, 'indUnk': 3287, '40': 1, 'Libra': 491, 'Publishing': 4, '36': 1708, '41': 20, '43': 6, 'HumanResources': 2, '14': 212, '15': 602, 'female': 4084, '24': 655, '45': 16, 'Sagittarius': 1097, 'Engineering': 127, 'Virgo': 236, 'Religion': 9, 'Technology': 2654, 'Pisces': 454, '34': 553, 'Aquarius': 571, '25': 386, 'Aries': 4198, 'Automotive': 14, 'Marketing': 156, 'Consulting': 21, 'Cancer': 504, '42': 14, 'InvestmentBanking': 70, 'Accounting': 4, 'Gemini': 150, '26': 234, 'Communications-Media': 99, 'male': 5916, 'Student': 1137, 'Sports-Recreation': 80, 'Taurus': 812, '44': 3, '39': 79, '46': 7, 'Science': 63, '27': 1054, '23': 253, 'BusinessServices': 91, 'Non-Profit': 71, '35': 2315, '16': 440}


### 7.Transform/Binarize the labels using MultiLabelBinarizer
As we have noticed before, in this task each example can have multiple tags. To deal with
such kind of prediction, we need to transform labels in a binary form and the prediction will be
a mask of 0s and 1s. For this purpose, it is convenient to use MultiLabelBinarizer from sklearn
a. Convert your train and test labels using MultiLabelBinarizer

In [34]:
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np
multilabel_binarizer = MultiLabelBinarizer()

multilabel_binarizer.fit(list(y_train['labels']))
print("CLASSES = ",multilabel_binarizer.classes_)

# transform target variable
#Train
multilabel_trainY = multilabel_binarizer.transform(list(y_train['labels']))
print("TRAIN SHAPE= ",(multilabel_trainY.shape))

#Test
multilabel_testY =  multilabel_binarizer.transform(list(y_test["labels"]))

print("TEST SHAPE= ",multilabel_testY.shape)

CLASSES =  ['13' '14' '15' '16' '17' '23' '24' '25' '26' '27' '33' '34' '35' '36'
 '37' '38' '39' '41' '42' '43' '44' '45' '46' 'Accounting' 'Aquarius'
 'Aries' 'Arts' 'Automotive' 'Banking' 'BusinessServices' 'Cancer'
 'Capricorn' 'Communications-Media' 'Consulting' 'Education' 'Engineering'
 'Fashion' 'Gemini' 'HumanResources' 'Internet' 'InvestmentBanking' 'Law'
 'LawEnforcement-Security' 'Leo' 'Libra' 'Marketing' 'Museums-Libraries'
 'Non-Profit' 'Pisces' 'Publishing' 'Religion' 'Sagittarius' 'Science'
 'Scorpio' 'Sports-Recreation' 'Student' 'Taurus' 'Technology'
 'Telecommunications' 'Virgo' 'female' 'indUnk' 'male']
TRAIN SHAPE=  (7500, 63)
TEST SHAPE=  (2500, 63)


### 8.Create a Classifier Model
Choose a classifier -
One-vs-Rest approach implemented in OneVsRestClassifier class is used . 
In this approach k classifiers (= number of tags) are trained. As a
basic classifier, use LogisticRegression

In [35]:
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier

#Model was not converging in default iteration of 100. So iterations increased to allow convergence 
lr_model = LogisticRegression(solver='lbfgs', max_iter = 200)
clf = OneVsRestClassifier(lr_model)

### 9.Fit the classifier, make predictions and get the accuracy 
a. Print the following
i. Accuracy score
ii. F1 score
iii. Average precision score
iv. Average recall score
v. Tip: Make sure you are familiar with all of them. How would you expect the
things to work for the multi-label scenario? Read about micro/macro/weighted
averaging

In [36]:
# fit model on train data
clf.fit(X_train_dtm, multilabel_trainY)
# make predictions for test set
y_pred = clf.predict(X_test_dtm)

In [37]:
#9.i ACCURACY SCORE
from sklearn import metrics

print("Train score ",metrics.accuracy_score(multilabel_trainY, clf.predict(X_train_dtm)))
print("Test accuracy score =",metrics.accuracy_score(multilabel_testY, y_pred))


Train score  0.9294666666666667
Test accuracy score = 0.3204


In [38]:
# 9.ii F1 SCORE
metrics.f1_score(multilabel_testY, y_pred, average="micro")

0.6469198409496917

In [39]:
#In multilabel confusion matrix , the count of true negatives is at 0,0, 
#false negatives is at [1,0], true positives is[1,1]  and false positives is [0,1].
metrics.multilabel_confusion_matrix(multilabel_testY, y_pred)

array([[[2483,    4],
        [   7,    6]],

       [[2437,    3],
        [  52,    8]],

       [[2328,   24],
        [ 105,   43]],

       [[2362,   22],
        [  82,   34]],

       [[2162,   45],
        [ 201,   92]],

       [[2435,   11],
        [  52,    2]],

       [[2334,   20],
        [ 119,   27]],

       [[2369,   11],
        [ 103,   17]],

       [[2434,    4],
        [  58,    4]],

       [[2230,   26],
        [ 150,   94]],

       [[2460,    0],
        [  23,   17]],

       [[2367,    2],
        [  40,   91]],

       [[1763,  150],
        [ 219,  368]],

       [[2047,   20],
        [ 177,  256]],

       [[2493,    0],
        [   7,    0]],

       [[2489,    0],
        [   6,    5]],

       [[2475,    3],
        [  21,    1]],

       [[2496,    0],
        [   4,    0]],

       [[2496,    0],
        [   4,    0]],

       [[2500,    0],
        [   0,    0]],

       [[2500,    0],
        [   0,    0]],

       [[2496,    0],
        [   

In [40]:
# 9.iii Average Precision Score
# 9.iv Average Recall Score
# Both Can be checked in classification report

In [41]:
# Print the precision and recall, among other metrics
print(metrics.classification_report(multilabel_testY, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.600     0.462     0.522        13
           1      0.727     0.133     0.225        60
           2      0.642     0.291     0.400       148
           3      0.607     0.293     0.395       116
           4      0.672     0.314     0.428       293
           5      0.154     0.037     0.060        54
           6      0.574     0.185     0.280       146
           7      0.607     0.142     0.230       120
           8      0.500     0.065     0.114        62
           9      0.783     0.385     0.516       244
          10      1.000     0.425     0.596        40
          11      0.978     0.695     0.813       131
          12      0.710     0.627     0.666       587
          13      0.928     0.591     0.722       433
          14      0.000     0.000     0.000         7
          15      1.000     0.455     0.625        11
          16      0.250     0.045     0.077        22
          17      0.000    

 ### 10.Print true label and predicted label for any five examples

In [42]:
for i in range(5):
    print("i=",i)
    print("Actual label binary = ",multilabel_testY[i])   
    print("\nActual label = ",multilabel_binarizer.inverse_transform(multilabel_testY)[i])
    print("\nPredicted label binary = ",y_pred[i])
    print("\nPredicted label = ",multilabel_binarizer.inverse_transform(y_pred)[i])
    print("\n")

i= 0
Actual label binary =  [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1]

Actual label =  ('35', 'Aries', 'Technology', 'male')

Predicted label binary =  [0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1]

Predicted label =  ('35', 'Aries', 'Technology', 'male')


i= 1
Actual label binary =  [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 1 0]

Actual label =  ('36', 'Pisces', 'female', 'indUnk')

Predicted label binary =  [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0]

Predicted label =  ('female', 'indUnk')


i= 2
Actual label binary =  [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1]

Actu

### Conclusions

1. Preprocessing was done by removing spaces, ',/, changing all to lower cases. Stop words are removed. This leads to reduction of unwanted text as our aim is to reduce feature space by removing unwanted and repeatable patterns


2. CountVectorizer was used to create document term matrix. Tuning was done for CountVectorizer by using n-gram, min_df,max_df, stemming and lemmatization. Best performance was obtained for n-gram= (1,2), min_df = 2, max_df=0.75, stemming as parameter to CountVectorizer.The tuning led to accodomation of 10000 blogs in memory of my system as well. So CountVectorizer tuning lead to better performance with comparatively less number of  features 
 
3. As number of blog samples were increased accuracy was reduced due to increase in number of features 

4.  Train score  0.9294666666666667
   Test accuracy score = 0.3204
5. 
    Micro, Macro, Weighted and samples average as per classification report are as follows:
    ---------------Precision--- Recall---F1  
    micro avg------0.763-------0.561-----0.647
    macro avg------0.497-------0.240-----0.305
    weighted avg---0.747-------0.561-----0.619
    samples avg----0.737-------0.561-----0.610

    In case of micro averaging precision and recall, all the individual True Positives, True Negatives, False Positives and False Negatives for each classes are summed up and their average is taken

    Micro averaged F1 score works extremely well when we have highly imbalanced distribution of tags

     The maximum micro averaged F1 score we have obtained from the entire project is 0.647 and the maximum value of weighted recall value that we have obtained is 0.561.

    Micro averaged F1 score should be our primary metric of choice simply because it takes the frequency of occurrence of tags into consideration.

     Given a very limited data size sample of 10000 datapoints, we have actually managed to get a decent micro averaged F1 score of 0.647
 
